# GPR Cavity/Utility Detection — Kaggle Baseline (YOLOv8)

**GPR subsurface cavity/void + utility detection baseline. NOT a landmine detector. Do NOT mix with the Thermal landmine dataset.**

Pipeline (mirrors `src/preprocessing/build_gpr_clean.py`):
audit → image/label validation → exact-dup removal → near-dup flag only → source-grouped leakage-safe split → schema standardization → `GPR_CLEAN` → YOLO.

The Mendeley dataset is already cropped/processed/augmented — this notebook never re-crops, resizes, denoises, enhances, or re-augments image data during preparation.

## 0. Environment: GPU + paths (Kaggle style)

In [ ]:
import sys, os, random
from pathlib import Path
print(sys.version)
try:
    import torch
    print('cuda:', torch.cuda.is_available())
except Exception as e:
    print('torch:', e)
print('cwd:', Path.cwd())
INROOT = Path('/kaggle/input')
print('kaggle input exists:', INROOT.exists())
if INROOT.exists():
    for p in sorted(INROOT.iterdir()):
        print('-', p)
SEED = 42
random.seed(SEED)
import numpy as np
np.random.seed(SEED)

## 1. Install YOLO (Kaggle: install once per session)

In [ ]:
%pip install -q ultralytics pillow pyyaml
from ultralytics import YOLO
import ultralytics; print('ultralytics', ultralytics.__version__)

## 2. Locate the GPR source — inspect BEFORE assuming structure
Supports an attached Kaggle dataset containing `GPR_data/` (preferred layout) or the flat legacy layout.

In [ ]:
# Optional: if you attached the dataset as a .zip, extract it once to /kaggle/working
from pathlib import Path
import zipfile
INROOT = Path('/kaggle/input')
zips = sorted(INROOT.rglob('*.zip')) if INROOT.exists() else []
print('zips found:', [str(z) for z in zips])
DEST = Path('/kaggle/working/_kaggle_src')
for z in zips:
    out = DEST / z.stem
    if not any(out.rglob('augmented_cavities')):
        out.mkdir(parents=True, exist_ok=True)
        print('extracting', z, '->', out)
        with zipfile.ZipFile(z) as zh: zh.extractall(out)
    else:
        print('already extracted:', out)
print('NOTE: prefer uploading the dataset unzipped; use .zip (not .rar) if you must compress.')


In [ ]:
from pathlib import Path
INROOT = Path('/kaggle/input')
DEST = Path('/kaggle/working/_kaggle_src')
roots = [INROOT] + ([DEST] if DEST.exists() else [])
cands = [p for r in roots for p in r.rglob('augmented_cavities')]
print('augmented_cavities hits:', [str(p) for p in cands])
if not cands:
    print('Top-level walk (depth<=3):')
    for r in roots:
        for p in sorted(r.rglob('*')):
            try: depth = len(p.relative_to(r).parts)
            except Exception: continue
            if depth <= 3:
                print(('  '*depth) + p.name)
SRC = cands[0].parent if cands else None
print('SRC =', SRC)
assert SRC is not None and SRC.exists(), 'Attach the GPR Kaggle dataset containing GPR_data/ and re-run.'


## 3. GPR pipeline logic (inline copy of `build_gpr_clean.py` — self-contained)

In [ ]:
import re, hashlib, io
CLASS_MAP = {'cavity': 0, 'utility': 1}
def validate_yolo_box(line):
    parts = line.strip().split()
    if len(parts) != 5: return False, f'malformed: {len(parts)} fields'
    try:
        float(parts[0]); cx, cy, w, h = map(float, parts[1:])
    except ValueError: return False, 'malformed: non-numeric'
    if not (0.0 <= cx <= 1.0 and 0.0 <= cy <= 1.0): return False, f'out-of-range center ({cx},{cy})'
    if not (0.0 < w <= 1.0 and 0.0 < h <= 1.0): return False, f'invalid size w={w} h={h}'
    return True, ''
def remap_label_lines(lines, kind):  # schema standardization, NOT a fix: folder carries semantics
    want = CLASS_MAP[kind]
    return [str(want) + ' ' + ' '.join(l.strip().split()[1:]) for l in lines]
def source_group_id(stem):
    from pathlib import Path as _P
    stem = _P(stem).stem
    m = re.match(r'(.+)_aug_\d+$', stem)
    return m.group(1) if m else stem
def normalize_stem(filename):
    from pathlib import Path as _P
    p = _P(filename)
    stem = re.sub(r'\s*\(\d+\)', lambda m: '_' + re.sub(r'\D', '', m.group(0)), p.stem)
    return re.sub(r'\s+', '_', stem.strip()) + '.jpg'
def dhash(b, hs=8):
    try:
        from PIL import Image
        im = Image.open(io.BytesIO(b)).convert('L').resize((hs+1, hs), Image.BILINEAR)
        px = list(im.get_flattened_data()) if hasattr(im, 'get_flattened_data') else list(im.getdata())
        bits = 0
        for r in range(hs):
            for c in range(hs):
                bits = (bits << 1) | (1 if px[r*(hs+1)+c] > px[r*(hs+1)+c+1] else 0)
        return bits
    except Exception: return None
print('pipeline fns ready')

## 4. Audit: collect items (read-only, bytes copied verbatim)

In [ ]:
from pathlib import Path
from collections import Counter
def find_label(root, kind, stem):
    if kind == 'intact': return None
    base = root / ('augmented_cavities' if kind == 'cavity' else 'augmented_utilities') / 'annotations'
    for sub in ['YOLO_format', 'Yolo_format', 'yolo_format']:
        c = base / sub / (stem + '.txt')
        if c.is_file(): return c.read_text()
    return None
items = []
for kind, dname in [('cavity','augmented_cavities'), ('utility','augmented_utilities'), ('intact','augmented_intact')]:
    for sub in ['images', '']:
        d = SRC / dname / sub if sub else SRC / dname
        if d.is_dir() and any(f.suffix.lower() in ('.jpg','.jpeg') for f in d.iterdir() if f.is_file()):
            for f in sorted(d.iterdir()):
                if f.is_file() and f.suffix.lower() in ('.jpg','.jpeg'):
                    items.append({'kind':kind,'ref':str(f),'bytes':f.read_bytes(),'base':f.stem,
                                  'label':find_label(SRC,kind,f.stem)})
            break
print('found:', Counter(i['kind'] for i in items), 'total:', len(items))

## 5. Validate: dims (report, never resize) + labels + duplicates

In [ ]:
import io, hashlib
from collections import Counter
from PIL import Image
dims = Counter(); bad_img = []; md5seen = {}; exact_dup = 0; valid = []
for it in items:
    try:
        im = Image.open(io.BytesIO(it['bytes'])); im.verify()
        im2 = Image.open(io.BytesIO(it['bytes']))
        dims[im2.size] += 1; it['size'] = im2.size
    except Exception:
        bad_img.append(it['ref']); continue
    h = hashlib.md5(it['bytes']).hexdigest()
    if h in md5seen: exact_dup += 1; it['drop']='exact-dup'; continue
    md5seen[h] = it['base']; valid.append(it)
print('dims (report only):', dict(dims))
print('corrupt:', len(bad_img), 'exact-dups removed:', exact_dup, 'kept:', len(valid))
# near-dup flags only (augmentation makes similarity legitimate)
def ham(a,b): return bin(a^b).count('1')
hashes = []
flags = 0
for it in valid:
    dh = dhash(it['bytes'])
    for ph,_n in hashes:
        if dh is not None and ham(dh,ph) <= 3:
            it['near_dup_flag'] = True; flags += 1; break
    hashes.append((dh, it['base']))
print('near-dup flags (kept, not deleted):', flags)
# label validation + standardization
invalid = []; pool = []
for it in valid:
    if it.get('drop'): continue
    if it['kind'] == 'intact': it['lines'] = []; pool.append(it); continue
    if not it['label']: invalid.append((it['ref'],'missing-label')); continue
    lines = [l for l in it['label'].strip().splitlines() if l.strip()]
    ok_all, clean = True, []
    for ln in lines:
        ok, why = validate_yolo_box(ln)
        if not ok: ok_all=False; invalid.append((it['ref'], why+' :: '+ln.strip())); break
        clean.append(ln)
    if not ok_all or not clean:
        if ok_all: invalid.append((it['ref'],'empty-label'))
        continue
    it['lines'] = remap_label_lines(clean, it['kind']); pool.append(it)
print('invalid labels:', len(invalid), invalid[:5])
print('pool:', Counter(i['kind'] for i in pool))

## 6. Group-aware split + leakage check + `GPR_CLEAN` + `data.yaml`

In [ ]:
import random, shutil, csv, json
from collections import Counter, defaultdict
from pathlib import Path
rng = random.Random(SEED)
OUT = Path('/kaggle/working/GPR_CLEAN')
if OUT.exists(): shutil.rmtree(OUT)
for s in ['train','val','test']:
    (OUT/'images'/s).mkdir(parents=True); (OUT/'labels'/s).mkdir(parents=True)
groups = defaultdict(list)
for it in pool: groups[(it['kind'], source_group_id(it['base']))].append(it)
assign = {}
for kind in ['cavity','utility','intact']:
    keys = sorted(g for (k,g) in groups if k==kind); rng.shuffle(keys)
    n=len(keys); nt, nv = int(n*0.7), int(n*0.15)
    for i,g in enumerate(keys): assign[(kind,g)] = 'train' if i<nt else ('val' if i<nt+nv else 'test')
for it in pool: it['split'] = assign[(it['kind'], source_group_id(it['base']))]
# leakage verification
seen = defaultdict(set)
for it in pool: seen[f"{it['kind']}:{source_group_id(it['base'])}"].add(it['split'])
leaked = [k for k,v in seen.items() if len(v)>1]
print('source groups:', len(seen), 'leaked groups (must be 0):', len(leaked))
assert len(leaked)==0, leaked[:5]
rows=[]; used=Counter()
for it in pool:
    nm = normalize_stem(it['base']+'.jpg')
    if used[nm]: nm = f'{Path(nm).stem}__{used[nm]}.jpg'
    used[nm]+=1
    (OUT/'images'/it['split']/nm).write_bytes(it['bytes'])
    (OUT/'labels'/it['split']/(Path(nm).stem+'.txt')).write_text('\n'.join(it['lines'])+('\n' if it['lines'] else ''))
    rows.append({'clean_path':f"images/{it['split']}/{nm}",'split':it['split'],'class':it['kind'],
                 'class_id':CLASS_MAP.get(it['kind'],-1),'source_ref':it['ref'],
                 'source_group':f"{it['kind']}:{source_group_id(it['base'])}",'orig_base':it['base'],
                 'md5':hashlib.md5(it['bytes']).hexdigest()})
with open(OUT/'manifest.csv','w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
with open(OUT/'data.yaml','w') as f:
    f.write(f'path: {OUT.as_posix()}\ntrain: images/train\nval: images/val\ntest: images/test\n'
            'nc: 2\nnames:\n  0: cavity\n  1: utility\n')
rep={'total':len(items),'kept':len(pool),'cavity':sum(1 for i in pool if i['kind']=='cavity'),
     'utility':sum(1 for i in pool if i['kind']=='utility'),'intact':sum(1 for i in pool if i['kind']=='intact'),
     'dims':{str(k):v for k,v in dims.items()},'invalid_labels':len(invalid),'exact_dups':exact_dup,
     'near_dup_flags':flags,'groups':len(seen),'leaked':len(leaked),
     'split':dict(Counter(i['split'] for i in pool)),
     'split_class':{f'{a}/{b}':c for (a,b),c in Counter((i['split'],i['kind']) for i in pool).items()}}
print(json.dumps(rep,indent=2))
(OUT/'cleaning_report.json').write_text(json.dumps(rep,indent=2))

## 7. Visual verification (boxes + dims; intact = no boxes)

In [ ]:
import random
from pathlib import Path
from PIL import Image, ImageDraw
random.seed(SEED)
OUT = Path('/kaggle/working/GPR_CLEAN')
def show(split, kind, n=3):
    import matplotlib.pyplot as plt
    labdir = OUT/'labels'/split; imgdir = OUT/'images'/split
    cands = [r for r in (OUT/'manifest.csv').read_text().splitlines()[1:] if f',{split},{kind},' in ','+r+',']
    random.shuffle(cands)
    for row in cands[:n]:
        rel = row.split(',')[0]
        im = Image.open(OUT/rel).convert('RGB')
        W,H = im.size; d = ImageDraw.Draw(im)
        lp = OUT/'labels'/split/(Path(rel).stem+'.txt')
        for ln in lp.read_text().splitlines():
            if not ln.strip(): continue
            c,x,y,w,h = map(float, ln.split())
            x0,y0,x1,y1 = (x-w/2)*W,(y-h/2)*H,(x+w/2)*W,(y+h/2)*H
            d.rectangle([x0,y0,x1,y1], outline='red', width=2)
            d.text((x0+2,y0+2), f"{'cavity' if int(c)==0 else 'utility'}", fill='red')
        plt.figure(); plt.imshow(im); plt.axis('off')
        plt.title(f'{kind} {split} {Path(rel).name} {W}x{H}')
        plt.show()
show('train','cavity'); show('train','utility'); show('train','intact')

## 8. YOLOv8 baseline (simple first experiment — no tuning, no extra augmentation)

In [ ]:
# Baseline model choice: 'yolov8n.pt' (fast) or 'yolov8m.pt' (heavier, slower). Start with n.
YOLO_MODEL = 'yolov8n.pt'  # change to 'yolov8m.pt' when ready
from pathlib import Path
from ultralytics import YOLO
OUT = Path('/kaggle/working/GPR_CLEAN')
CFG = {'model': YOLO_MODEL,'epochs':50,'imgsz':224,'batch':16,'seed':42}
print(CFG, 'data:', str(OUT/'data.yaml'))
model = YOLO(CFG['model'])
res = model.train(data=str(OUT/'data.yaml'), epochs=CFG['epochs'], imgsz=CFG['imgsz'],
                  batch=CFG['batch'], seed=CFG['seed'], workers=2, verbose=True)
print('train done:', model.ckpt_path if hasattr(model,'ckpt_path') else 'check runs/detect/train')


## 9. Results: metrics, per-class, confusion matrix

In [ ]:
from pathlib import Path
from ultralytics import YOLO
best = sorted(Path('/kaggle/working').rglob('weights/best.pt'))
print('checkpoints:', [str(p) for p in best])
m = YOLO(str(best[-1]) if best else 'yolov8n.pt')
for split in ['val','test']:
    print('='*30, split)
    print(m.val(data='/kaggle/working/GPR_CLEAN/data.yaml', split=split))
import glob
for p in ['runs/detect/train/results.png','runs/detect/train/confusion_matrix.png',
          'runs/detect/val/confusion_matrix.png']:
    hits = glob.glob('/kaggle/working/'+p)+glob.glob(p)
    print(p, hits[:1])
    if hits:
        from PIL import Image; import matplotlib.pyplot as plt
        plt.figure(); plt.imshow(Image.open(hits[0])); plt.axis('off'); plt.title(p); plt.show()